# 03 — LayoutLMv3 fine-tuning (low-VRAM, RTX 3050 4 GB)
`microsoft/layoutlmv3-base` + `LayoutLMv3ForTokenClassification`, full fine-tuning (no LoRA, no freezing).

> **Honesty note:** this notebook ships **without outputs**. Every number you see after running it is produced by the code on your machine. Nothing is pre-filled.

In [ ]:
# Run from anywhere inside the repo: move to project root so src/ imports and relative paths work
import os, sys
from pathlib import Path
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))
print("Project root:", root)

## 6. GPU detection

In [ ]:
from src.utils.hardware import print_hardware
hw = print_hardware()

## 1–2. Processor and label mappings

In [ ]:
from src.extraction.training import load_processed
from src.extraction.layoutlmv3_model import load_processor, build_model
ds, labels = load_processed()
label2id = {l: i for i, l in enumerate(labels)}; id2label = {i: l for l, i in label2id.items()}
processor = load_processor()   # apply_ocr=False: we supply words + boxes ourselves
print(type(processor).__name__); print(label2id)

## 3. Dataset encoding (one document → one or more 256-token chunks)

In [ ]:
from src.extraction.layoutlmv3_model import word_token_counts, chunk_ranges, prepare_inputs, encode_chunk
doc = ds["train"][0]
counts = word_token_counts(processor.tokenizer, doc["words"])
chunks = chunk_ranges(counts, 256)
print("words:", len(doc["words"]), "| chunks:", chunks)
img, boxes = prepare_inputs(doc["image"], doc["boxes"], no_layout=False)
pix = processor.image_processor(img, return_tensors="pt")["pixel_values"]   # resized to 224x224
s, e = chunks[0]
enc = encode_chunk(processor.tokenizer, pix, doc["words"][s:e], boxes[s:e],
                   [label2id[l] for l in doc["labels"][s:e]], max_length=256)
{k: tuple(v.shape) for k, v in enc.items()}

## 4–5. Model loading and parameter check (must be > 10M trainable)

In [ ]:
from src.utils.parameters import check_parameters
model = build_model(labels)
report = check_parameters(model, report_path="results/parameter_report.txt")
del model   # the training script builds its own copy; avoid two models in VRAM
import gc, torch; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 7–10. Training setup, training, validation, best checkpoint
Defaults: batch 1, grad-accum 8, max_length 256, fp16, gradient checkpointing, image 224, num_workers 0. Validation is carved out of the train split; the test split is not touched. The best macro-F1 checkpoint is saved to `models/layoutlmv3/best`.

Running from a terminal is more robust on Windows (the notebook kernel can hold GPU memory). If you get CUDA OOM, lower `--max-length` to 192.

In [ ]:
!python scripts/train_layoutlmv3.py --epochs 5 --batch-size 1 --gradient-accumulation 8 --max-length 256 --learning-rate 5e-5 --fp16 --gradient-checkpointing

In [ ]:
import json, pandas as pd
log = json.load(open("results/training_log.json"))
hist = pd.DataFrame(log["history"]) if isinstance(log, dict) and "history" in log else pd.DataFrame(log)
hist

In [ ]:
import os
print(os.listdir("models/layoutlmv3/best"))
print(open("results/parameter_report.txt").read())